# RhythmMamba Environment Installation Guide

This guide creates the working RhythmMamba environment for:

- Linux x86-64
- Python 3.11
- NVIDIA Tesla V100
- PyTorch 2.1.2 with CUDA 12.1
- `causal-conv1d 1.4.0`
- `mamba-ssm 2.2.2`

> Run all installation commands in a terminal, not in a Python/Jupyter cell.

---

## Step 1: Create the Conda environment

```bash
conda create -n mamba_hunting python=3.11 pip -y
conda activate mamba_hunting
```

Confirm that the correct Python is active:

```bash
which python
python --version
```

Expected:

```text
.../miniconda3/envs/mamba_hunting/bin/python
Python 3.11.x
```

---

## Step 2: Install compatible build tools

PyTorch 2.1.2 uses the older `pkg_resources.packaging` interface. Therefore, `setuptools==69.5.1` is required.

```bash
python -m pip install --upgrade pip

python -m pip install \
  setuptools==69.5.1 \
  wheel==0.43.0 \
  packaging \
  ninja
```

Verify:

```bash
python -c "from pkg_resources import packaging; print('Compatible packaging:', packaging.__version__)"
```

A `pkg_resources is deprecated` warning is harmless.

---

## Step 3: Install compatible NumPy and SciPy

NumPy 2.x is incompatible with this PyTorch build. Use NumPy 1.26.4.

```bash
python -m pip install \
  numpy==1.26.4 \
  scipy==1.11.4
```

Verify:

```bash
python -c "import numpy, scipy; print('NumPy:',numpy.__version__); print('SciPy:',scipy.__version__)"
```

Expected:

```text
NumPy: 1.26.4
SciPy: 1.11.4
```

---

## Step 4: Install PyTorch with CUDA 12.1

Use the official PyTorch pip wheels. This avoids the Conda MKL `iJIT_NotifyEvent` error.

```bash
python -m pip install \
  torch==2.1.2 \
  torchvision==0.16.2 \
  torchaudio==2.1.2 \
  --index-url https://download.pytorch.org/whl/cu121
```

Verify PyTorch and CUDA:

```bash
python -c "import torch; print('PyTorch:',torch.__version__); print('CUDA:',torch.version.cuda); print('CUDA available:',torch.cuda.is_available()); print('GPU:',torch.cuda.get_device_name(0))"
```

Expected:

```text
PyTorch: 2.1.2+cu121
CUDA: 12.1
CUDA available: True
GPU: Tesla V100-PCIE-32GB
```

The `Can't initialize NVML` warning affects GPU monitoring only. It does not prevent CUDA computation.

---

## Step 5: Install Mamba Python dependencies

`transformers` must be pinned because newer versions removed APIs required by `mamba-ssm 2.2.2`.

```bash
python -m pip install \
  einops==0.7.0 \
  transformers==4.39.3
```

---

## Step 6: Install the precompiled CUDA extensions

Do not install `causal-conv1d` or `mamba-ssm` directly from their source packages. Source compilation requires the CUDA compiler `nvcc`.

### Install causal-conv1d

```bash
python -m pip install --no-deps \
  "https://github.com/Dao-AILab/causal-conv1d/releases/download/v1.4.0/causal_conv1d-1.4.0%2Bcu122torch2.1cxx11abiFALSE-cp311-cp311-linux_x86_64.whl"
```

Verify:

```bash
python -c "import causal_conv1d; print('causal-conv1d:', causal_conv1d.__version__)"
```

Expected:

```text
causal-conv1d: 1.4.0
```

### Install mamba-ssm

```bash
python -m pip install --no-deps \
  "https://github.com/state-spaces/mamba/releases/download/v2.2.2/mamba_ssm-2.2.2%2Bcu122torch2.1cxx11abiFALSE-cp311-cp311-linux_x86_64.whl"
```

Verify:

```bash
python -c "import mamba_ssm; print('mamba-ssm:', mamba_ssm.__version__)"
```

Expected:

```text
mamba-ssm: 2.2.2
```

The `cu122` wheels are the official CUDA-12 builds and are compatible with the PyTorch CUDA 12.1 runtime.

---

## Step 7: Install the remaining RhythmMamba dependencies

The original `requirements.txt` contains several versions that are too old for Python 3.11. Use these compatible versions:

```bash
python -m pip install \
  h5py==3.10.0 \
  pandas==2.1.4 \
  scikit-image==0.22.0 \
  matplotlib==3.8.2 \
  opencv-python==4.8.1.78 \
  PyYAML==6.0.1 \
  scikit-learn==1.3.2 \
  tensorboardX==2.6.2.2 \
  timm==0.9.16 \
  tqdm==4.66.3 \
  mat73==0.59 \
  yacs==0.1.8 \
  thop==0.1.1.post2209072238
```

> Do not run the official repository's complete `requirements.txt` inside this Python 3.11 environment.

---

## Step 8: Install and register the Jupyter kernel

Install the notebook packages:

```bash
python -m pip install \
  ipykernel==6.29.5 \
  ipywidgets==8.1.1
```

Register the environment as a Jupyter kernel:

```bash
python -m ipykernel install \
  --user \
  --name mamba_hunting \
  --display-name "Python (Mamba Hunting)"
```

In VS Code, select:

```text
Python (Mamba Hunting)
```

Installing `ipywidgets` prevents the `IProgress not found` warning. Restart the notebook kernel after installation.

---

## Step 9: Verify the complete environment

### Check all important package versions

```bash
python -c "import numpy, scipy, torch, causal_conv1d, mamba_ssm, transformers; print('NumPy:',numpy.__version__); print('SciPy:',scipy.__version__); print('PyTorch:',torch.__version__); print('CUDA:',torch.version.cuda); print('causal-conv1d:',causal_conv1d.__version__); print('mamba-ssm:',mamba_ssm.__version__); print('transformers:',transformers.__version__)"
```

Expected principal versions:

```text
NumPy: 1.26.4
SciPy: 1.11.4
PyTorch: 2.1.2+cu121
CUDA: 12.1
causal-conv1d: 1.4.0
mamba-ssm: 2.2.2
transformers: 4.39.3
```

### Check dependency consistency

```bash
python -m pip check
```

### Run a Mamba CUDA operation test

```bash
python -c "import torch; from mamba_ssm.modules.mamba_simple import Mamba; model=Mamba(d_model=96,d_state=48,d_conv=4,expand=2).cuda(); x=torch.randn(1,80,96,device='cuda'); y=model(x); print('Input:',x.shape); print('Output:',y.shape); print('Mamba CUDA test: PASSED')"
```

Expected:

```text
Input: torch.Size([1, 80, 96])
Output: torch.Size([1, 80, 96])
Mamba CUDA test: PASSED
```

If this test passes, the Mamba CUDA environment is ready to run RhythmMamba.

In [3]:
from pathlib import Path
import os
import sys
import time

import torch

# ============================================================
# LOCATE THE OFFICIAL CODE
# ============================================================

PROJECT_ROOT = Path(
    "/media/data/rPPG/Code/GitHub/Catch_The_Mamba"
)

OFFICIAL_ROOT = PROJECT_ROOT / "official" / "RhythmMamba"

assert OFFICIAL_ROOT.is_dir(), (
    f"Official RhythmMamba folder not found: {OFFICIAL_ROOT}"
)

os.chdir(OFFICIAL_ROOT)

if str(OFFICIAL_ROOT) not in sys.path:
    sys.path.insert(0, str(OFFICIAL_ROOT))

from neural_methods.model.RhythmMamba import RhythmMamba


# ============================================================
# GPU CHECK
# ============================================================

assert torch.cuda.is_available(), "CUDA GPU is not available"

device = torch.device("cuda:0")
torch.cuda.set_device(device)

print("=" * 70)
print("ENVIRONMENT")
print("=" * 70)

print("PyTorch version :", torch.__version__)
print("CUDA version    :", torch.version.cuda)
print("Selected GPU    :", torch.cuda.get_device_name(device))
print("Official code   :", OFFICIAL_ROOT)


# ============================================================
# CREATE THE COMPLETE MODEL
# ============================================================

model = RhythmMamba().to(device)
model.eval()

trainable_parameters = sum(
    parameter.numel()
    for parameter in model.parameters()
    if parameter.requires_grad
)

total_parameters = sum(
    parameter.numel()
    for parameter in model.parameters()
)

print("\n" + "=" * 70)
print("MODEL INFORMATION")
print("=" * 70)

print(f"Trainable parameters : {trainable_parameters:,}")
print(f"Trainable parameters : {trainable_parameters / 1e6:.3f} M")
print(f"Total parameters     : {total_parameters:,}")


# ============================================================
# ARTIFICIAL 160-FRAME VIDEO
# Shape: [batch, time, RGB, height, width]
# ============================================================

test_input = torch.randn(
    1, 160, 3, 128, 128,
    device=device,
    dtype=torch.float32,
)

torch.cuda.reset_peak_memory_stats(device)
torch.cuda.synchronize(device)

start_time = time.perf_counter()

with torch.inference_mode():
    prediction = model(test_input)

torch.cuda.synchronize(device)
elapsed_time = time.perf_counter() - start_time

peak_memory = torch.cuda.max_memory_allocated(device) / 1024**3


# ============================================================
# VERIFY OUTPUT
# ============================================================

print("\n" + "=" * 70)
print("FORWARD-PASS RESULT")
print("=" * 70)

print("Input shape        :", tuple(test_input.shape))
print("Prediction shape   :", tuple(prediction.shape))
print("Prediction finite  :", bool(torch.isfinite(prediction).all()))
print(f"Inference time     : {elapsed_time:.4f} seconds")
print(f"Peak GPU memory    : {peak_memory:.3f} GB")

assert prediction.shape == (1, 160)
assert torch.isfinite(prediction).all()

print("\nComplete official RhythmMamba test: PASSED")

ENVIRONMENT
PyTorch version : 2.1.2+cu121
CUDA version    : 12.1
Selected GPU    : Tesla V100-PCIE-32GB
Official code   : /media/data/rPPG/Code/GitHub/Catch_The_Mamba/official/RhythmMamba

MODEL INFORMATION
Trainable parameters : 4,922,125
Trainable parameters : 4.922 M
Total parameters     : 4,922,125

FORWARD-PASS RESULT
Input shape        : (1, 160, 3, 128, 128)
Prediction shape   : (1, 160)
Prediction finite  : True
Inference time     : 0.0525 seconds
Peak GPU memory    : 0.437 GB

Complete official RhythmMamba test: PASSED


# Dataet Validation (for PURE and UBFC-rPPG)

In [9]:
from pathlib import Path
import sys
import shutil
import cv2

# ============================================================
# PATHS
# ============================================================

PROJECT_ROOT = Path(
    "/media/data/rPPG/Code/GitHub/Catch_The_Mamba"
)

OFFICIAL_ROOT = PROJECT_ROOT / "official" / "RhythmMamba"

PURE_PATH = Path(
    "/media/data/rPPG/rPPG_Data/Mamba_Hunt/RhythmMamba_DataView/PURE"
)

UBFC_PATH = Path(
    "/media/data/rPPG/rPPG_Data/Mamba_Hunt/RhythmMamba_DataView/UBFC"
)

assert PROJECT_ROOT.exists(), f"Project directory not found: {PROJECT_ROOT}"
assert OFFICIAL_ROOT.exists(), f"Official code not found: {OFFICIAL_ROOT}"
assert PURE_PATH.exists(), f"PURE compatibility view not found: {PURE_PATH}"
assert UBFC_PATH.exists(), f"UBFC compatibility view not found: {UBFC_PATH}"

if str(OFFICIAL_ROOT) not in sys.path:
    sys.path.insert(0, str(OFFICIAL_ROOT))


# ============================================================
# SCIPY COMPATIBILITY FIX
# ============================================================
#
# The official data_loader/__init__.py automatically imports
# every dataset loader, including the unused MMPD loader.
#
# MMPDLoader imports scipy.__config__.get_info, which does not
# exist in modern SciPy. The imported function is not actually
# used by MMPDLoader.
#
# This compatibility shim allows the official package to import.
# It does not load, process, or use the MMPD dataset.
# ============================================================

import scipy.__config__ as scipy_config

if not hasattr(scipy_config, "get_info"):
    scipy_config.get_info = lambda *args, **kwargs: {}


# Import only the loaders that we will use.
from dataset.data_loader.PURELoader import PURELoader
from dataset.data_loader.UBFCrPPGLoader import UBFCrPPGLoader


# ============================================================
# PURE OFFICIAL-LOADER CHECK
# ============================================================

pure_loader = PURELoader.__new__(PURELoader)
pure_loader.dataset_name = "PURE"

pure_records = sorted(
    pure_loader.get_raw_data(str(PURE_PATH)),
    key=lambda item: item["index"],
)

assert len(pure_records) > 0, "No PURE recordings were detected."

pure_record = pure_records[0]
pure_record_path = Path(pure_record["path"])
pure_name = pure_record_path.name

# Layout expected by the official PURE loader:
# PURE/01-01/01-01/*.png
# PURE/01-01/01-01.json

pure_image_directory = pure_record_path / pure_name
pure_json_path = pure_record_path / f"{pure_name}.json"

assert pure_image_directory.exists(), (
    f"PURE image directory missing: {pure_image_directory}"
)

assert pure_json_path.exists(), (
    f"PURE JSON file missing: {pure_json_path}"
)

pure_images = sorted(pure_image_directory.glob("*.png"))

assert len(pure_images) > 0, (
    f"No PNG frames found in: {pure_image_directory}"
)

pure_first_image = cv2.imread(str(pure_images[0]))

assert pure_first_image is not None, (
    f"Could not read PURE image: {pure_images[0]}"
)

pure_wave = PURELoader.read_wave(str(pure_json_path))

assert len(pure_wave) > 0, "The PURE waveform is empty."


# ============================================================
# UBFC-rPPG OFFICIAL-LOADER CHECK
# ============================================================

ubfc_loader = UBFCrPPGLoader.__new__(UBFCrPPGLoader)
ubfc_loader.dataset_name = "UBFC"

ubfc_records = ubfc_loader.get_raw_data(str(UBFC_PATH))

assert len(ubfc_records) > 0, "No UBFC recordings were detected."

ubfc_record = next(
    (
        record
        for record in ubfc_records
        if record["index"] == "subject1"
    ),
    ubfc_records[0],
)

ubfc_record_path = Path(ubfc_record["path"])

# Layout expected by the official UBFC loader:
# UBFC/subject1/vid.avi
# UBFC/subject1/ground_truth.txt

ubfc_video_path = ubfc_record_path / "vid.avi"
ubfc_label_path = ubfc_record_path / "ground_truth.txt"

assert ubfc_video_path.exists(), (
    f"UBFC video missing: {ubfc_video_path}"
)

assert ubfc_label_path.exists(), (
    f"UBFC ground-truth file missing: {ubfc_label_path}"
)

ubfc_wave = UBFCrPPGLoader.read_wave(str(ubfc_label_path))

assert len(ubfc_wave) > 0, "The UBFC waveform is empty."

video = cv2.VideoCapture(str(ubfc_video_path))

assert video.isOpened(), (
    f"OpenCV could not open the UBFC video: {ubfc_video_path}"
)

ubfc_frame_count = int(video.get(cv2.CAP_PROP_FRAME_COUNT))
ubfc_fps = float(video.get(cv2.CAP_PROP_FPS))
ubfc_width = int(video.get(cv2.CAP_PROP_FRAME_WIDTH))
ubfc_height = int(video.get(cv2.CAP_PROP_FRAME_HEIGHT))

success, ubfc_first_frame = video.read()
video.release()

assert success, "Could not read the first UBFC video frame."
assert ubfc_first_frame is not None, "The first UBFC frame is empty."


# ============================================================
# AVAILABLE STORAGE
# ============================================================

disk_information = shutil.disk_usage("/media/data")

total_storage_gb = disk_information.total / (1024**3)
used_storage_gb = disk_information.used / (1024**3)
free_storage_gb = disk_information.free / (1024**3)


# ============================================================
# RESULTS
# ============================================================

print("=" * 70)
print("PURE OFFICIAL-LOADER CHECK")
print("=" * 70)
print("Recordings found :", len(pure_records))
print("Test recording   :", pure_name)
print("Image directory  :", pure_image_directory)
print("JSON file        :", pure_json_path)
print("PNG frames       :", len(pure_images))
print("First frame shape:", pure_first_image.shape)
print("Waveform samples :", len(pure_wave))

print("\n" + "=" * 70)
print("UBFC-rPPG OFFICIAL-LOADER CHECK")
print("=" * 70)
print("Recordings found :", len(ubfc_records))
print("Test recording   :", ubfc_record["index"])
print("Video file       :", ubfc_video_path)
print("Label file       :", ubfc_label_path)
print("Video frames     :", ubfc_frame_count)
print("Video FPS        :", ubfc_fps)
print("Video resolution :", (ubfc_width, ubfc_height))
print("First frame shape:", ubfc_first_frame.shape)
print("Waveform samples :", len(ubfc_wave))

print("\n" + "=" * 70)
print("STORAGE INFORMATION")
print("=" * 70)
print(f"Total storage    : {total_storage_gb:.2f} GB")
print(f"Used storage     : {used_storage_gb:.2f} GB")
print(f"Free storage     : {free_storage_gb:.2f} GB")

print("\nOfficial PURE and UBFC-rPPG loader validation: PASSED")

PURE OFFICIAL-LOADER CHECK
Recordings found : 59
Test recording   : 01-01
Image directory  : /media/data/rPPG/rPPG_Data/RhythmMamba_DataView/PURE/01-01/01-01
JSON file        : /media/data/rPPG/rPPG_Data/RhythmMamba_DataView/PURE/01-01/01-01.json
PNG frames       : 2026
First frame shape: (480, 640, 3)
Waveform samples : 4018

UBFC-rPPG OFFICIAL-LOADER CHECK
Recordings found : 42
Test recording   : subject1
Video file       : /media/data/rPPG/rPPG_Data/RhythmMamba_DataView/UBFC/subject1/vid.avi
Label file       : /media/data/rPPG/rPPG_Data/RhythmMamba_DataView/UBFC/subject1/ground_truth.txt
Video frames     : 1547
Video FPS        : 29.264106
Video resolution : (640, 480)
First frame shape: (480, 640, 3)
Waveform samples : 1547

STORAGE INFORMATION
Total storage    : 1150.24 GB
Used storage     : 798.40 GB
Free storage     : 293.34 GB

Official PURE and UBFC-rPPG loader validation: PASSED


## preprocessing on only one PURE recording before processing the complete dataset

In [11]:
from pathlib import Path
from types import SimpleNamespace
import os
import sys
import numpy as np
import scipy.__config__ as scipy_config

# ============================================================
# PATHS
# ============================================================

PROJECT_ROOT = Path(
    "/media/data/rPPG/Code/GitHub/Catch_The_Mamba"
)

OFFICIAL_ROOT = PROJECT_ROOT / "official" / "RhythmMamba"

PURE_PATH = Path(
    "/media/data/rPPG/rPPG_Data/Mamba_Hunt/RhythmMamba_DataView/PURE"
)

SMOKE_CACHE = Path(
    "/media/data/rPPG/rPPG_Data/"
    "RhythmMamba_Preprocessed_Smoke/PURE_01-01"
)

OFFICIAL_CONFIG = (
    OFFICIAL_ROOT
    / "configs"
    / "train_configs"
    / "intra"
    / "1PURE_RHYTHMMAMBA.yaml"
)

if str(OFFICIAL_ROOT) not in sys.path:
    sys.path.insert(0, str(OFFICIAL_ROOT))


# ============================================================
# COMPATIBILITY FIX FOR UNUSED MMPD IMPORT
# ============================================================

if not hasattr(scipy_config, "get_info"):
    scipy_config.get_info = lambda *args, **kwargs: {}


# ============================================================
# IMPORT OFFICIAL CODE
# ============================================================

from config import get_config
from dataset.data_loader.PURELoader import PURELoader


# ============================================================
# LOAD OFFICIAL PREPROCESSING SETTINGS
# ============================================================

config = get_config(
    SimpleNamespace(config_file=str(OFFICIAL_CONFIG))
)

preprocess_config = config.TRAIN.DATA.PREPROCESS

print("=" * 70)
print("OFFICIAL PREPROCESSING SETTINGS")
print("=" * 70)
print("Data type       :", preprocess_config.DATA_TYPE)
print("Label type      :", preprocess_config.LABEL_TYPE)
print("Chunk length    :", preprocess_config.CHUNK_LENGTH)
print("Crop face       :", preprocess_config.CROP_FACE.DO_CROP_FACE)
print("Large face box  :", preprocess_config.CROP_FACE.USE_LARGE_FACE_BOX)
print("Output size     :", (
    preprocess_config.RESIZE.H,
    preprocess_config.RESIZE.W,
))


# ============================================================
# PREPARE OFFICIAL LOADER
# ============================================================

loader = PURELoader.__new__(PURELoader)

# Normally created by BaseLoader.__init__.
# They must be initialized because we intentionally bypassed
# the constructor to process only one recording.
loader.inputs = []
loader.labels = []
loader.inputs_data = []
loader.labels_data = []
loader.incremental_train = 0
loader.incremental_rate = 0.5

loader.dataset_name = "PURE-smoke"
loader.raw_data_path = str(PURE_PATH)
loader.cached_path = str(SMOKE_CACHE)
loader.config_data = config.TRAIN.DATA

pure_records = sorted(
    loader.get_raw_data(str(PURE_PATH)),
    key=lambda item: item["index"],
)

selected_records = [
    record
    for record in pure_records
    if Path(record["path"]).name == "01-01"
]

assert len(selected_records) == 1, (
    f"Expected one PURE 01-01 recording, found {len(selected_records)}"
)

SMOKE_CACHE.mkdir(parents=True, exist_ok=True)


# ============================================================
# PREPROCESS ONE RECORDING
# ============================================================

existing_inputs = sorted(SMOKE_CACHE.glob("*_input*.npy"))

if existing_inputs:
    print("\nExisting smoke-test cache found; preprocessing skipped.")
    input_files = [str(path) for path in existing_inputs]

else:
    file_list_dictionary = {}

    # Face detection uses a relative path inside the official repository.
    previous_directory = Path.cwd()

    try:
        os.chdir(OFFICIAL_ROOT)

        loader.preprocess_dataset_subprocess(
            data_dirs=selected_records,
            config_preprocess=preprocess_config,
            i=0,
            file_list_dict=file_list_dictionary,
        )

    finally:
        os.chdir(previous_directory)

    input_files = file_list_dictionary[0]


# ============================================================
# VALIDATE GENERATED CLIPS
# ============================================================

input_files = sorted(input_files)
label_files = [
    input_path.replace("_input", "_label")
    for input_path in input_files
]

assert len(input_files) > 0, "No input clips were generated."
assert len(input_files) == len(label_files)

first_input = np.load(input_files[0], mmap_mode="r")
first_label = np.load(label_files[0], mmap_mode="r")

assert first_input.shape == (160, 128, 128, 3)
assert first_label.shape == (160,)
assert np.isfinite(first_input).all()
assert np.isfinite(first_label).all()

cache_size_bytes = sum(
    path.stat().st_size
    for path in SMOKE_CACHE.glob("*.npy")
)

cache_size_gb = cache_size_bytes / (1024**3)


# ============================================================
# RESULTS
# ============================================================

print("\n" + "=" * 70)
print("PURE PREPROCESSING SMOKE TEST")
print("=" * 70)
print("Recording          : 01-01")
print("Generated clips    :", len(input_files))
print("Input clip shape   :", first_input.shape)
print("Label clip shape   :", first_label.shape)
print("Input dtype        :", first_input.dtype)
print("Label dtype        :", first_label.dtype)
print("Input finite       :", bool(np.isfinite(first_input).all()))
print("Label finite       :", bool(np.isfinite(first_label).all()))
print(f"Smoke-cache size   : {cache_size_gb:.3f} GB")
print("Cache directory    :", SMOKE_CACHE)

print("\nOfficial PURE preprocessing smoke test: PASSED")

=> Merging a config file from /media/data/rPPG/Code/GitHub/Catch_The_Mamba/official/RhythmMamba/configs/train_configs/intra/1PURE_RHYTHMMAMBA.yaml
OFFICIAL PREPROCESSING SETTINGS
Data type       : ['Standardized']
Label type      : Standardized
Chunk length    : 160
Crop face       : True
Large face box  : True
Output size     : (128, 128)

PURE PREPROCESSING SMOKE TEST
Recording          : 01-01
Generated clips    : 12
Input clip shape   : (160, 128, 128, 3)
Label clip shape   : (160,)
Input dtype        : float64
Label dtype        : float64
Input finite       : True
Label finite       : True
Smoke-cache size   : 0.703 GB
Cache directory    : /media/data/rPPG/rPPG_Data/RhythmMamba_Preprocessed_Smoke/PURE_01-01

Official PURE preprocessing smoke test: PASSED


## test one UBFC-rPPG recording using the same official preprocessing pipeline:

In [12]:
from pathlib import Path
from types import SimpleNamespace
import os
import sys
import numpy as np
import scipy.__config__ as scipy_config

# ============================================================
# PATHS
# ============================================================

PROJECT_ROOT = Path(
    "/media/data/rPPG/Code/GitHub/Catch_The_Mamba"
)

OFFICIAL_ROOT = PROJECT_ROOT / "official" / "RhythmMamba"

UBFC_PATH = Path(
    "/media/data/rPPG/rPPG_Data/Mamba_Hunt/RhythmMamba_DataView/UBFC"
)

SMOKE_CACHE = Path(
    "/media/data/rPPG/rPPG_Data/"
    "RhythmMamba_Preprocessed_Smoke/UBFC_subject1"
)

OFFICIAL_CONFIG = (
    OFFICIAL_ROOT
    / "configs"
    / "train_configs"
    / "intra"
    / "2UBFC-rPPG_RHYTHMMAMBA.yaml"
)

assert OFFICIAL_ROOT.exists()
assert UBFC_PATH.exists()
assert OFFICIAL_CONFIG.exists()

if str(OFFICIAL_ROOT) not in sys.path:
    sys.path.insert(0, str(OFFICIAL_ROOT))


# ============================================================
# COMPATIBILITY FIX FOR UNUSED MMPD IMPORT
# ============================================================

if not hasattr(scipy_config, "get_info"):
    scipy_config.get_info = lambda *args, **kwargs: {}


# ============================================================
# IMPORT OFFICIAL CODE
# ============================================================

from config import get_config
from dataset.data_loader.UBFCrPPGLoader import UBFCrPPGLoader


# ============================================================
# LOAD OFFICIAL PREPROCESSING SETTINGS
# ============================================================

config = get_config(
    SimpleNamespace(config_file=str(OFFICIAL_CONFIG))
)

preprocess_config = config.TRAIN.DATA.PREPROCESS

print("=" * 70)
print("OFFICIAL UBFC PREPROCESSING SETTINGS")
print("=" * 70)
print("Data type       :", preprocess_config.DATA_TYPE)
print("Label type      :", preprocess_config.LABEL_TYPE)
print("Chunk length    :", preprocess_config.CHUNK_LENGTH)
print("Crop face       :", preprocess_config.CROP_FACE.DO_CROP_FACE)
print("Large face box  :", preprocess_config.CROP_FACE.USE_LARGE_FACE_BOX)
print(
    "Output size     :",
    (
        preprocess_config.RESIZE.H,
        preprocess_config.RESIZE.W,
    ),
)


# ============================================================
# PREPARE OFFICIAL LOADER
# ============================================================

loader = UBFCrPPGLoader.__new__(UBFCrPPGLoader)

# Normally initialized by BaseLoader.__init__.
loader.inputs = []
loader.labels = []
loader.inputs_data = []
loader.labels_data = []
loader.incremental_train = 0
loader.incremental_rate = 0.5

loader.dataset_name = "UBFC-smoke"
loader.raw_data_path = str(UBFC_PATH)
loader.cached_path = str(SMOKE_CACHE)
loader.config_data = config.TRAIN.DATA

ubfc_records = loader.get_raw_data(str(UBFC_PATH))

selected_records = [
    record
    for record in ubfc_records
    if record["index"] == "subject1"
]

assert len(selected_records) == 1, (
    f"Expected one UBFC subject1 recording, "
    f"found {len(selected_records)}"
)

SMOKE_CACHE.mkdir(parents=True, exist_ok=True)


# ============================================================
# PREPROCESS SUBJECT 1
# ============================================================

existing_inputs = sorted(
    SMOKE_CACHE.glob("*_input*.npy")
)

if existing_inputs:
    print("\nExisting UBFC smoke-test cache found.")
    print("Preprocessing skipped.")

    input_files = [
        str(path)
        for path in existing_inputs
    ]

else:
    file_list_dictionary = {}
    previous_directory = Path.cwd()

    try:
        # Required for the official Haar-cascade relative path.
        os.chdir(OFFICIAL_ROOT)

        loader.preprocess_dataset_subprocess(
            data_dirs=selected_records,
            config_preprocess=preprocess_config,
            i=0,
            file_list_dict=file_list_dictionary,
        )

    finally:
        os.chdir(previous_directory)

    input_files = file_list_dictionary[0]


# ============================================================
# VALIDATE GENERATED CLIPS
# ============================================================

input_files = sorted(input_files)

label_files = [
    input_path.replace("_input", "_label")
    for input_path in input_files
]

assert len(input_files) > 0
assert len(input_files) == len(label_files)

first_input = np.load(
    input_files[0],
    mmap_mode="r",
)

first_label = np.load(
    label_files[0],
    mmap_mode="r",
)

assert first_input.shape == (160, 128, 128, 3)
assert first_label.shape == (160,)
assert np.isfinite(first_input).all()
assert np.isfinite(first_label).all()

cache_size_bytes = sum(
    path.stat().st_size
    for path in SMOKE_CACHE.glob("*.npy")
)

cache_size_gb = cache_size_bytes / (1024**3)


# ============================================================
# RESULTS
# ============================================================

print("\n" + "=" * 70)
print("UBFC-rPPG PREPROCESSING SMOKE TEST")
print("=" * 70)
print("Recording          : subject1")
print("Generated clips    :", len(input_files))
print("Input clip shape   :", first_input.shape)
print("Label clip shape   :", first_label.shape)
print("Input dtype        :", first_input.dtype)
print("Label dtype        :", first_label.dtype)
print("Input finite       :", bool(np.isfinite(first_input).all()))
print("Label finite       :", bool(np.isfinite(first_label).all()))
print(f"Smoke-cache size   : {cache_size_gb:.3f} GB")
print("Cache directory    :", SMOKE_CACHE)

print("\nOfficial UBFC-rPPG preprocessing smoke test: PASSED")

=> Merging a config file from /media/data/rPPG/Code/GitHub/Catch_The_Mamba/official/RhythmMamba/configs/train_configs/intra/2UBFC-rPPG_RHYTHMMAMBA.yaml
OFFICIAL UBFC PREPROCESSING SETTINGS
Data type       : ['Standardized']
Label type      : Standardized
Chunk length    : 160
Crop face       : True
Large face box  : True
Output size     : (128, 128)

UBFC-rPPG PREPROCESSING SMOKE TEST
Recording          : subject1
Generated clips    : 9
Input clip shape   : (160, 128, 128, 3)
Label clip shape   : (160,)
Input dtype        : float64
Label dtype        : float64
Input finite       : True
Label finite       : True
Smoke-cache size   : 0.527 GB
Cache directory    : /media/data/rPPG/rPPG_Data/RhythmMamba_Preprocessed_Smoke/UBFC_subject1

Official UBFC-rPPG preprocessing smoke test: PASSED


In [13]:
from pathlib import Path

# ============================================================
# CENTRAL RHYTHMMAMBA WORK DIRECTORY
# ============================================================

MAMBA_HUNT_ROOT = Path(
    "/media/data/rPPG/rPPG_Data/Mamba_Hunt"
)

DATA_VIEW_ROOT = (
    MAMBA_HUNT_ROOT / "RhythmMamba_DataView"
)

SMOKE_CACHE_ROOT = (
    MAMBA_HUNT_ROOT / "RhythmMamba_Preprocessed_Smoke"
)

PREPROCESSED_ROOT = (
    MAMBA_HUNT_ROOT / "RhythmMamba_Preprocessed"
)

PURE_VIEW = DATA_VIEW_ROOT / "PURE"
UBFC_VIEW = DATA_VIEW_ROOT / "UBFC"

PURE_SMOKE = SMOKE_CACHE_ROOT / "PURE_01-01"
UBFC_SMOKE = SMOKE_CACHE_ROOT / "UBFC_subject1"


# Create only the empty destination for full preprocessing.
PREPROCESSED_ROOT.mkdir(parents=True, exist_ok=True)


# ============================================================
# VERIFY MOVED DIRECTORIES
# ============================================================

assert MAMBA_HUNT_ROOT.exists(), MAMBA_HUNT_ROOT
assert DATA_VIEW_ROOT.exists(), DATA_VIEW_ROOT
assert SMOKE_CACHE_ROOT.exists(), SMOKE_CACHE_ROOT
assert PREPROCESSED_ROOT.exists(), PREPROCESSED_ROOT

assert PURE_VIEW.exists(), PURE_VIEW
assert UBFC_VIEW.exists(), UBFC_VIEW

pure_recordings = sorted(PURE_VIEW.glob("*-*"))
ubfc_recordings = sorted(UBFC_VIEW.glob("subject*"))

assert len(pure_recordings) == 59, (
    f"Expected 59 PURE recordings, found {len(pure_recordings)}"
)

assert len(ubfc_recordings) == 42, (
    f"Expected 42 UBFC recordings, found {len(ubfc_recordings)}"
)


# ============================================================
# VERIFY REPRESENTATIVE SYMLINK TARGETS
# ============================================================

pure_images = PURE_VIEW / "01-01" / "01-01"
pure_json = PURE_VIEW / "01-01" / "01-01.json"

ubfc_video = UBFC_VIEW / "subject1" / "vid.avi"
ubfc_label = UBFC_VIEW / "subject1" / "ground_truth.txt"

assert pure_images.exists(), pure_images
assert pure_json.exists(), pure_json
assert ubfc_video.exists(), ubfc_video
assert ubfc_label.exists(), ubfc_label


# ============================================================
# VERIFY SMOKE-CACHE FILES
# ============================================================

pure_smoke_inputs = list(PURE_SMOKE.glob("*_input*.npy"))
ubfc_smoke_inputs = list(UBFC_SMOKE.glob("*_input*.npy"))

assert len(pure_smoke_inputs) == 12, (
    f"Expected 12 PURE smoke clips, found {len(pure_smoke_inputs)}"
)

assert len(ubfc_smoke_inputs) == 9, (
    f"Expected 9 UBFC smoke clips, found {len(ubfc_smoke_inputs)}"
)


# ============================================================
# RESULTS
# ============================================================

print("=" * 70)
print("MAMBA HUNT DIRECTORY VERIFICATION")
print("=" * 70)
print("Main directory       :", MAMBA_HUNT_ROOT)
print("PURE recordings      :", len(pure_recordings))
print("UBFC recordings      :", len(ubfc_recordings))
print("PURE smoke clips     :", len(pure_smoke_inputs))
print("UBFC smoke clips     :", len(ubfc_smoke_inputs))
print("Full cache directory :", PREPROCESSED_ROOT)

print("\nResolved original targets:")
print("PURE images          :", pure_images.resolve())
print("PURE JSON            :", pure_json.resolve())
print("UBFC video           :", ubfc_video.resolve())
print("UBFC label           :", ubfc_label.resolve())

print("\nMamba Hunt directory migration: PASSED")

MAMBA HUNT DIRECTORY VERIFICATION
Main directory       : /media/data/rPPG/rPPG_Data/Mamba_Hunt
PURE recordings      : 59
UBFC recordings      : 42
PURE smoke clips     : 12
UBFC smoke clips     : 9
Full cache directory : /media/data/rPPG/rPPG_Data/Mamba_Hunt/RhythmMamba_Preprocessed

Resolved original targets:
PURE images          : /media/data/rPPG/rPPG_Data/PURE/ALL/ALL/01-01
PURE JSON            : /media/data/rPPG/rPPG_Data/PURE/ALL/ALL/01-01.json
UBFC video           : /media/data/rPPG/rPPG_Data/UBFC_rPPG/vid_1/vid_1.avi
UBFC label           : /media/data/rPPG/rPPG_Data/UBFC_rPPG/vid_1/ground_truth_1.txt

Mamba Hunt directory migration: PASSED


# create the two local configuration files

In [14]:
from pathlib import Path
import yaml

# ============================================================
# PATHS
# ============================================================

PROJECT_ROOT = Path(
    "/media/data/rPPG/Code/GitHub/Catch_The_Mamba"
)

OFFICIAL_ROOT = (
    PROJECT_ROOT / "official" / "RhythmMamba"
)

MAMBA_HUNT_ROOT = Path(
    "/media/data/rPPG/rPPG_Data/Mamba_Hunt"
)

DATA_VIEW_ROOT = (
    MAMBA_HUNT_ROOT / "RhythmMamba_DataView"
)

PREPROCESSED_ROOT = (
    MAMBA_HUNT_ROOT / "RhythmMamba_Preprocessed"
)

PURE_DATA_PATH = DATA_VIEW_ROOT / "PURE"
UBFC_DATA_PATH = DATA_VIEW_ROOT / "UBFC"

LOCAL_CONFIG_DIRECTORY = (
    PROJECT_ROOT / "configs" / "local" / "intra"
)

RESULTS_ROOT = PROJECT_ROOT / "results"


# ============================================================
# CREATE REQUIRED DIRECTORIES
# ============================================================

LOCAL_CONFIG_DIRECTORY.mkdir(
    parents=True,
    exist_ok=True,
)

PREPROCESSED_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)

(RESULTS_ROOT / "models").mkdir(
    parents=True,
    exist_ok=True,
)

(RESULTS_ROOT / "runs").mkdir(
    parents=True,
    exist_ok=True,
)

(RESULTS_ROOT / "predictions").mkdir(
    parents=True,
    exist_ok=True,
)


# ============================================================
# CONFIGURATION CREATION FUNCTION
# ============================================================

def create_local_config(
    source_config: Path,
    destination_config: Path,
    dataset_name: str,
    data_path: Path,
    cache_path: Path,
):
    with source_config.open(
        "r",
        encoding="utf-8",
    ) as file:
        configuration = yaml.safe_load(file)

    # Update paths for every dataset split.
    for split_name in ["TRAIN", "VALID", "TEST"]:
        data_configuration = configuration[split_name]["DATA"]

        data_configuration["DATASET"] = dataset_name
        data_configuration["DATA_PATH"] = str(data_path)
        data_configuration["CACHED_PATH"] = str(cache_path)

        # Full preprocessing will be handled separately.
        # Training must only read the completed cache.
        data_configuration["DO_PREPROCESS"] = False

    # Keep experiment outputs inside the user's GitHub project.
    configuration["LOG"]["PATH"] = str(
        RESULTS_ROOT / "runs"
    )

    configuration["MODEL"]["MODEL_DIR"] = str(
        RESULTS_ROOT / "models"
    )

    configuration["TEST"]["OUTPUT_SAVE_DIR"] = str(
        RESULTS_ROOT / "predictions"
    )

    with destination_config.open(
        "w",
        encoding="utf-8",
    ) as file:
        yaml.safe_dump(
            configuration,
            file,
            sort_keys=False,
        )


# ============================================================
# PURE CONFIGURATION
# ============================================================

pure_official_config = (
    OFFICIAL_ROOT
    / "configs"
    / "train_configs"
    / "intra"
    / "1PURE_RHYTHMMAMBA.yaml"
)

pure_local_config = (
    LOCAL_CONFIG_DIRECTORY
    / "PURE_RHYTHMMAMBA_LOCAL.yaml"
)

create_local_config(
    source_config=pure_official_config,
    destination_config=pure_local_config,
    dataset_name="PURE",
    data_path=PURE_DATA_PATH,
    cache_path=PREPROCESSED_ROOT / "PURE",
)


# ============================================================
# UBFC-rPPG CONFIGURATION
# ============================================================

ubfc_official_config = (
    OFFICIAL_ROOT
    / "configs"
    / "train_configs"
    / "intra"
    / "2UBFC-rPPG_RHYTHMMAMBA.yaml"
)

ubfc_local_config = (
    LOCAL_CONFIG_DIRECTORY
    / "UBFC_RHYTHMMAMBA_LOCAL.yaml"
)

create_local_config(
    source_config=ubfc_official_config,
    destination_config=ubfc_local_config,

    # Official main.py recognizes "UBFC", not "UBFC-rPPG".
    dataset_name="UBFC",

    data_path=UBFC_DATA_PATH,
    cache_path=PREPROCESSED_ROOT / "UBFC",
)


# ============================================================
# VERIFY LOCAL CONFIGURATIONS
# ============================================================

for config_path in [
    pure_local_config,
    ubfc_local_config,
]:
    assert config_path.exists()

    with config_path.open(
        "r",
        encoding="utf-8",
    ) as file:
        configuration = yaml.safe_load(file)

    dataset = configuration["TRAIN"]["DATA"]

    assert Path(dataset["DATA_PATH"]).exists()
    assert dataset["DO_PREPROCESS"] is False
    assert dataset["PREPROCESS"]["CHUNK_LENGTH"] == 160
    assert dataset["PREPROCESS"]["RESIZE"]["H"] == 128
    assert dataset["PREPROCESS"]["RESIZE"]["W"] == 128
    assert dataset["PREPROCESS"]["DATA_TYPE"] == ["Standardized"]
    assert dataset["PREPROCESS"]["LABEL_TYPE"] == "Standardized"

    print("=" * 70)
    print("LOCAL CONFIGURATION")
    print("=" * 70)
    print("Configuration :", config_path)
    print("Dataset       :", dataset["DATASET"])
    print("Dataset path  :", dataset["DATA_PATH"])
    print("Cache root    :", dataset["CACHED_PATH"])
    print("Preprocessing :", dataset["DO_PREPROCESS"])
    print("Data type     :", dataset["PREPROCESS"]["DATA_TYPE"])
    print("Label type    :", dataset["PREPROCESS"]["LABEL_TYPE"])
    print("Chunk length  :", dataset["PREPROCESS"]["CHUNK_LENGTH"])
    print(
        "Frame size    :",
        (
            dataset["PREPROCESS"]["RESIZE"]["H"],
            dataset["PREPROCESS"]["RESIZE"]["W"],
        ),
    )
    print()


print("Local PURE and UBFC configurations created successfully.")
print("Official RhythmMamba submodule remains unchanged.")

LOCAL CONFIGURATION
Configuration : /media/data/rPPG/Code/GitHub/Catch_The_Mamba/configs/local/intra/PURE_RHYTHMMAMBA_LOCAL.yaml
Dataset       : PURE
Dataset path  : /media/data/rPPG/rPPG_Data/Mamba_Hunt/RhythmMamba_DataView/PURE
Cache root    : /media/data/rPPG/rPPG_Data/Mamba_Hunt/RhythmMamba_Preprocessed/PURE
Preprocessing : False
Data type     : ['Standardized']
Label type    : Standardized
Chunk length  : 160
Frame size    : (128, 128)

LOCAL CONFIGURATION
Configuration : /media/data/rPPG/Code/GitHub/Catch_The_Mamba/configs/local/intra/UBFC_RHYTHMMAMBA_LOCAL.yaml
Dataset       : UBFC
Dataset path  : /media/data/rPPG/rPPG_Data/Mamba_Hunt/RhythmMamba_DataView/UBFC
Cache root    : /media/data/rPPG/rPPG_Data/Mamba_Hunt/RhythmMamba_Preprocessed/UBFC
Preprocessing : False
Data type     : ['Standardized']
Label type    : Standardized
Chunk length  : 160
Frame size    : (128, 128)

Local PURE and UBFC configurations created successfully.
Official RhythmMamba submodule remains unchanged.


# preprocess the complete PURE

This step:

Reads all 59 PURE recordings.<br>
Uses the official preprocessing.<br>
Creates 160-frame, 128×128 standardized clips.<br>
Uses four CPU processes.<br>
Shows recording-level progress.<br>
Supports rerunning by skipping completed recordings.<br>
Does not modify the original PURE dataset.<br>

In [15]:
from pathlib import Path
from types import SimpleNamespace
import gc
import os
import shutil
import sys

import numpy as np
import pandas as pd
import scipy.__config__ as scipy_config

# ============================================================
# SETTINGS
# ============================================================

PROJECT_ROOT = Path(
    "/media/data/rPPG/Code/GitHub/Catch_The_Mamba"
)

OFFICIAL_ROOT = PROJECT_ROOT / "official" / "RhythmMamba"

LOCAL_CONFIG = (
    PROJECT_ROOT
    / "configs"
    / "local"
    / "intra"
    / "PURE_RHYTHMMAMBA_LOCAL.yaml"
)

MAX_PROCESSES = 4

assert OFFICIAL_ROOT.exists()
assert LOCAL_CONFIG.exists()

if str(OFFICIAL_ROOT) not in sys.path:
    sys.path.insert(0, str(OFFICIAL_ROOT))


# ============================================================
# COMPATIBILITY FIX FOR UNUSED MMPD IMPORT
# ============================================================

if not hasattr(scipy_config, "get_info"):
    scipy_config.get_info = lambda *args, **kwargs: {}


# ============================================================
# IMPORT OFFICIAL CODE
# ============================================================

from config import get_config
from dataset.data_loader.PURELoader import PURELoader


# ============================================================
# LOAD LOCAL CONFIGURATION
# ============================================================

config = get_config(
    SimpleNamespace(config_file=str(LOCAL_CONFIG))
)

data_config = config.TRAIN.DATA
preprocess_config = data_config.PREPROCESS

raw_data_path = Path(data_config.DATA_PATH)
cache_path = Path(data_config.CACHED_PATH)
chunk_length = preprocess_config.CHUNK_LENGTH

assert raw_data_path.exists()

cache_path.mkdir(
    parents=True,
    exist_ok=True,
)


# ============================================================
# PREPARE OFFICIAL PURE LOADER
# ============================================================

loader = PURELoader.__new__(PURELoader)

loader.inputs = []
loader.labels = []
loader.inputs_data = []
loader.labels_data = []
loader.incremental_train = 0
loader.incremental_rate = 0.5

loader.dataset_name = "PURE"
loader.raw_data_path = str(raw_data_path)
loader.cached_path = str(cache_path)
loader.config_data = data_config

pure_records = sorted(
    loader.get_raw_data(str(raw_data_path)),
    key=lambda item: item["index"],
)

assert len(pure_records) == 59


# ============================================================
# CALCULATE EXPECTED CLIPS AND RESUME STATUS
# ============================================================

def expected_pure_clips(record):
    record_path = Path(record["path"])
    recording_name = record_path.name
    image_directory = record_path / recording_name

    frame_count = len(
        list(image_directory.glob("*.png"))
    )

    return frame_count // chunk_length


def recording_is_complete(record):
    recording_id = record["index"]
    expected_clips = expected_pure_clips(record)

    input_files = list(
        cache_path.glob(f"{recording_id}_input*.npy")
    )

    label_files = list(
        cache_path.glob(f"{recording_id}_label*.npy")
    )

    if len(input_files) != expected_clips:
        return False

    if len(label_files) != expected_clips:
        return False

    return all(
        file_path.stat().st_size > 0
        for file_path in input_files + label_files
    )


completed_before_start = [
    record
    for record in pure_records
    if recording_is_complete(record)
]

records_to_process = [
    record
    for record in pure_records
    if not recording_is_complete(record)
]

expected_total_clips = sum(
    expected_pure_clips(record)
    for record in pure_records
)

estimated_bytes = expected_total_clips * (
    (160 * 128 * 128 * 3 * 8)  # Input float64
    + (160 * 8)                # Label float64
)

estimated_size_gb = estimated_bytes / (1024**3)

disk_before = shutil.disk_usage(
    cache_path.parent
)


# ============================================================
# PREPROCESSING SUMMARY
# ============================================================

print("=" * 70)
print("FULL PURE PREPROCESSING")
print("=" * 70)
print("Raw dataset          :", raw_data_path)
print("Cache directory      :", cache_path)
print("Total recordings     :", len(pure_records))
print("Already completed    :", len(completed_before_start))
print("Remaining recordings :", len(records_to_process))
print("Expected clips       :", expected_total_clips)
print(f"Estimated cache size : {estimated_size_gb:.2f} GB")
print(
    f"Free storage before  : "
    f"{disk_before.free / (1024**3):.2f} GB"
)
print("Parallel processes   :", MAX_PROCESSES)


# ============================================================
# RUN OFFICIAL PREPROCESSING
# ============================================================

if records_to_process:
    previous_directory = Path.cwd()

    try:
        # Required by the official Haar-cascade relative path.
        os.chdir(OFFICIAL_ROOT)

        loader.multi_process_manager(
            data_dirs=records_to_process,
            config_preprocess=preprocess_config,
            multi_process_quota=MAX_PROCESSES,
        )

    finally:
        os.chdir(previous_directory)

else:
    print("\nAll PURE recordings are already processed.")


# ============================================================
# VALIDATE EVERY RECORDING
# ============================================================

failed_records = [
    Path(record["path"]).name
    for record in pure_records
    if not recording_is_complete(record)
]

if failed_records:
    raise RuntimeError(
        "Incomplete PURE recordings: "
        + ", ".join(failed_records)
        + "\nRerun this cell to retry only incomplete recordings."
    )


# ============================================================
# CREATE OFFICIAL TRAIN AND TEST FILE LISTS
# ============================================================

split_information = [
    (
        "TRAIN",
        config.TRAIN.DATA.BEGIN,
        config.TRAIN.DATA.END,
        Path(config.TRAIN.DATA.FILE_LIST_PATH),
    ),
    (
        "TEST",
        config.TEST.DATA.BEGIN,
        config.TEST.DATA.END,
        Path(config.TEST.DATA.FILE_LIST_PATH),
    ),
]

split_counts = {}

for split_name, begin, end, file_list_path in split_information:
    loader.file_list_path = str(file_list_path)

    loader.build_file_list_retroactive(
        data_dirs=pure_records,
        begin=begin,
        end=end,
    )

    file_list = pd.read_csv(file_list_path)
    input_files = file_list["input_files"].tolist()

    assert len(input_files) > 0
    assert all(Path(path).exists() for path in input_files)
    assert all(
        Path(path.replace("_input", "_label")).exists()
        for path in input_files
    )

    split_counts[split_name] = len(input_files)


# ============================================================
# FINAL CACHE VALIDATION
# ============================================================

all_inputs = sorted(
    cache_path.glob("*_input*.npy")
)

all_labels = sorted(
    cache_path.glob("*_label*.npy")
)

assert len(all_inputs) == expected_total_clips
assert len(all_labels) == expected_total_clips

sample_input = np.load(
    all_inputs[0],
    mmap_mode="r",
)

sample_label = np.load(
    all_labels[0],
    mmap_mode="r",
)

assert sample_input.shape == (160, 128, 128, 3)
assert sample_label.shape == (160,)
assert np.isfinite(sample_input).all()
assert np.isfinite(sample_label).all()

cache_size_bytes = sum(
    path.stat().st_size
    for path in all_inputs + all_labels
)

disk_after = shutil.disk_usage(
    cache_path.parent
)

gc.collect()


# ============================================================
# RESULTS
# ============================================================

print("\n" + "=" * 70)
print("FULL PURE PREPROCESSING RESULT")
print("=" * 70)
print("Recordings processed :", len(pure_records))
print("Total input clips    :", len(all_inputs))
print("Total label clips    :", len(all_labels))
print("Training clips       :", split_counts["TRAIN"])
print("Testing clips        :", split_counts["TEST"])
print("Sample input shape   :", sample_input.shape)
print("Sample label shape   :", sample_label.shape)
print("Input dtype          :", sample_input.dtype)
print("Label dtype          :", sample_label.dtype)
print(
    f"Actual cache size    : "
    f"{cache_size_bytes / (1024**3):.2f} GB"
)
print(
    f"Free storage after   : "
    f"{disk_after.free / (1024**3):.2f} GB"
)
print("Train file list      :", config.TRAIN.DATA.FILE_LIST_PATH)
print("Test file list       :", config.TEST.DATA.FILE_LIST_PATH)

print("\nComplete PURE preprocessing: PASSED")

=> Merging a config file from /media/data/rPPG/Code/GitHub/Catch_The_Mamba/configs/local/intra/PURE_RHYTHMMAMBA_LOCAL.yaml
FULL PURE PREPROCESSING
Raw dataset          : /media/data/rPPG/rPPG_Data/Mamba_Hunt/RhythmMamba_DataView/PURE
Cache directory      : /media/data/rPPG/rPPG_Data/Mamba_Hunt/RhythmMamba_Preprocessed/PURE/PURE_SizeW128_SizeH128_ClipLength160_DataTypeStandardized_DataAugNone_LabelTypeStandardized_Crop_faceTrue_Large_boxTrue_Large_size1.5_Dyamic_DetFalse_det_len30_Median_face_boxFalse
Total recordings     : 59
Already completed    : 0
Remaining recordings : 59
Expected clips       : 750
Estimated cache size : 43.95 GB
Free storage before  : 290.88 GB
Parallel processes   : 4
Preprocessing dataset...


 10%|█         | 6/59 [01:01<07:11,  8.14s/it]

 17%|█▋        | 10/59 [01:35<06:16,  7.69s/it]

 24%|██▎       | 14/59 [02:06<04:58,  6.64s/it]

 31%|███       | 18/59 [02:38<03:32,  5.17s/it]

...

 92%|█████████▏| 54/59 [07:52<00:32,  6.48s/it]

 93%|█████████▎| 55/59 [08:14<00:44, 11.10s/it]

 97%|█████████▋| 57/59 [08:32<00:18,  9.32s/it]

100%|██████████| 59/59 [08:44<00:00,  8.89s/it]



FULL PURE PREPROCESSING RESULT
Recordings processed : 59
Total input clips    : 750
Total label clips    : 750
Training clips       : 443
Testing clips        : 307
Sample input shape   : (160, 128, 128, 3)
Sample label shape   : (160,)
Input dtype          : float64
Label dtype          : float64
Actual cache size    : 43.95 GB
Free storage after   : 246.93 GB
Train file list      : /media/data/rPPG/rPPG_Data/Mamba_Hunt/RhythmMamba_Preprocessed/PURE/DataFileLists/PURE_SizeW128_SizeH128_ClipLength160_DataTypeStandardized_DataAugNone_LabelTypeStandardized_Crop_faceTrue_Large_boxTrue_Large_size1.5_Dyamic_DetFalse_det_len30_Median_face_boxFalse_0.0_0.6.csv
Test file list       : /media/data/rPPG/rPPG_Data/Mamba_Hunt/RhythmMamba_Preprocessed/PURE/DataFileLists/PURE_SizeW128_SizeH128_ClipLength160_DataTypeStandardized_DataAugNone_LabelTypeStandardized_Crop_faceTrue_Large_boxTrue_Large_size1.5_Dyamic_DetFalse_det_len30_Median_face_boxFalse_0.6_1.0.csv

Complete PURE preprocessing: PASSED


# complete UBFC-rPPG preprocessing

In [16]:
from pathlib import Path
from types import SimpleNamespace
import gc
import os
import shutil
import sys

import cv2
import numpy as np
import pandas as pd
import scipy.__config__ as scipy_config

# ============================================================
# SETTINGS
# ============================================================

PROJECT_ROOT = Path(
    "/media/data/rPPG/Code/GitHub/Catch_The_Mamba"
)

OFFICIAL_ROOT = PROJECT_ROOT / "official" / "RhythmMamba"

LOCAL_CONFIG = (
    PROJECT_ROOT
    / "configs"
    / "local"
    / "intra"
    / "UBFC_RHYTHMMAMBA_LOCAL.yaml"
)

MAX_PROCESSES = 4

assert OFFICIAL_ROOT.exists()
assert LOCAL_CONFIG.exists()

if str(OFFICIAL_ROOT) not in sys.path:
    sys.path.insert(0, str(OFFICIAL_ROOT))


# ============================================================
# COMPATIBILITY FIX FOR UNUSED MMPD IMPORT
# ============================================================

if not hasattr(scipy_config, "get_info"):
    scipy_config.get_info = lambda *args, **kwargs: {}


# ============================================================
# IMPORT OFFICIAL CODE
# ============================================================

from config import get_config
from dataset.data_loader.UBFCrPPGLoader import UBFCrPPGLoader


# ============================================================
# LOAD LOCAL CONFIGURATION
# ============================================================

config = get_config(
    SimpleNamespace(config_file=str(LOCAL_CONFIG))
)

data_config = config.TRAIN.DATA
preprocess_config = data_config.PREPROCESS

raw_data_path = Path(data_config.DATA_PATH)
cache_path = Path(data_config.CACHED_PATH)
chunk_length = preprocess_config.CHUNK_LENGTH

assert raw_data_path.exists()

# TRAIN and TEST must use the same generated cache.
assert Path(config.TEST.DATA.CACHED_PATH) == cache_path

cache_path.mkdir(
    parents=True,
    exist_ok=True,
)


# ============================================================
# PREPARE OFFICIAL UBFC LOADER
# ============================================================

loader = UBFCrPPGLoader.__new__(UBFCrPPGLoader)

loader.inputs = []
loader.labels = []
loader.inputs_data = []
loader.labels_data = []
loader.incremental_train = 0
loader.incremental_rate = 0.5

loader.dataset_name = "UBFC"
loader.raw_data_path = str(raw_data_path)
loader.cached_path = str(cache_path)
loader.config_data = data_config

# Keep the official loader's ordering because the official
# train/test split is based on this ordering.
ubfc_records = loader.get_raw_data(
    str(raw_data_path)
)

assert len(ubfc_records) == 42


# ============================================================
# READ VIDEO/LABEL METADATA
# ============================================================

recording_information = {}

for record in ubfc_records:
    recording_id = record["index"]
    recording_path = Path(record["path"])

    video_path = recording_path / "vid.avi"
    label_path = recording_path / "ground_truth.txt"

    assert video_path.exists(), video_path
    assert label_path.exists(), label_path

    video = cv2.VideoCapture(str(video_path))

    assert video.isOpened(), (
        f"Could not open video: {video_path}"
    )

    frame_count = int(
        video.get(cv2.CAP_PROP_FRAME_COUNT)
    )

    fps = float(
        video.get(cv2.CAP_PROP_FPS)
    )

    video.release()

    assert frame_count > 0, (
        f"Invalid video frame count: {video_path}"
    )

    waveform = loader.read_wave(
        str(label_path)
    )

    label_count = len(waveform)
    expected_clips = frame_count // chunk_length

    # Each generated label clip must contain 160 values.
    assert label_count >= expected_clips * chunk_length, (
        f"{recording_id}: insufficient labels. "
        f"Frames={frame_count}, labels={label_count}"
    )

    recording_information[recording_id] = {
        "frames": frame_count,
        "labels": label_count,
        "fps": fps,
        "expected_clips": expected_clips,
    }


# ============================================================
# RESUME CHECK
# ============================================================

def recording_is_complete(record):
    recording_id = record["index"]

    expected_clips = recording_information[
        recording_id
    ]["expected_clips"]

    input_files = list(
        cache_path.glob(
            f"{recording_id}_input*.npy"
        )
    )

    label_files = list(
        cache_path.glob(
            f"{recording_id}_label*.npy"
        )
    )

    if len(input_files) != expected_clips:
        return False

    if len(label_files) != expected_clips:
        return False

    return all(
        file_path.stat().st_size > 0
        for file_path in input_files + label_files
    )


completed_before_start = [
    record
    for record in ubfc_records
    if recording_is_complete(record)
]

records_to_process = [
    record
    for record in ubfc_records
    if not recording_is_complete(record)
]

expected_total_clips = sum(
    information["expected_clips"]
    for information in recording_information.values()
)

estimated_bytes = expected_total_clips * (
    (160 * 128 * 128 * 3 * 8)
    + (160 * 8)
)

estimated_size_gb = estimated_bytes / (1024**3)

disk_before = shutil.disk_usage(
    cache_path.parent
)

frame_label_differences = [
    (
        recording_id,
        information["frames"],
        information["labels"],
    )
    for recording_id, information
    in recording_information.items()
    if information["frames"] != information["labels"]
]


# ============================================================
# PREPROCESSING SUMMARY
# ============================================================

print("=" * 70)
print("FULL UBFC-rPPG PREPROCESSING")
print("=" * 70)
print("Raw dataset          :", raw_data_path)
print("Cache directory      :", cache_path)
print("Total recordings     :", len(ubfc_records))
print("Already completed    :", len(completed_before_start))
print("Remaining recordings :", len(records_to_process))
print("Expected clips       :", expected_total_clips)
print(f"Estimated cache size : {estimated_size_gb:.2f} GB")
print(
    f"Free storage before  : "
    f"{disk_before.free / (1024**3):.2f} GB"
)
print("Parallel processes   :", MAX_PROCESSES)
print(
    "Frame/label mismatch:",
    len(frame_label_differences),
)

if frame_label_differences:
    print("\nRecordings with different frame/label counts:")

    for recording_id, frames, labels in frame_label_differences:
        print(
            f"  {recording_id}: "
            f"frames={frames}, labels={labels}"
        )


# ============================================================
# RUN OFFICIAL PREPROCESSING
# ============================================================

if records_to_process:
    previous_directory = Path.cwd()

    try:
        # Required for the official Haar-cascade path.
        os.chdir(OFFICIAL_ROOT)

        loader.multi_process_manager(
            data_dirs=records_to_process,
            config_preprocess=preprocess_config,
            multi_process_quota=MAX_PROCESSES,
        )

    finally:
        os.chdir(previous_directory)

else:
    print("\nAll UBFC recordings are already processed.")


# ============================================================
# VALIDATE EVERY RECORDING
# ============================================================

failed_records = [
    record["index"]
    for record in ubfc_records
    if not recording_is_complete(record)
]

if failed_records:
    raise RuntimeError(
        "Incomplete UBFC recordings: "
        + ", ".join(failed_records)
        + "\nRerun this cell to retry only incomplete recordings."
    )


# ============================================================
# CREATE OFFICIAL TRAIN AND TEST FILE LISTS
# ============================================================

split_information = [
    (
        "TRAIN",
        config.TRAIN.DATA.BEGIN,
        config.TRAIN.DATA.END,
        Path(config.TRAIN.DATA.FILE_LIST_PATH),
    ),
    (
        "TEST",
        config.TEST.DATA.BEGIN,
        config.TEST.DATA.END,
        Path(config.TEST.DATA.FILE_LIST_PATH),
    ),
]

split_counts = {}

for split_name, begin, end, file_list_path in split_information:
    loader.file_list_path = str(file_list_path)

    loader.build_file_list_retroactive(
        data_dirs=ubfc_records,
        begin=begin,
        end=end,
    )

    file_list = pd.read_csv(
        file_list_path
    )

    input_files = file_list[
        "input_files"
    ].tolist()

    assert len(input_files) > 0
    assert all(
        Path(path).exists()
        for path in input_files
    )
    assert all(
        Path(
            path.replace("_input", "_label")
        ).exists()
        for path in input_files
    )

    split_counts[split_name] = len(
        input_files
    )


# ============================================================
# FINAL CACHE VALIDATION
# ============================================================

all_inputs = sorted(
    cache_path.glob("*_input*.npy")
)

all_labels = sorted(
    cache_path.glob("*_label*.npy")
)

assert len(all_inputs) == expected_total_clips
assert len(all_labels) == expected_total_clips

sample_input = np.load(
    all_inputs[0],
    mmap_mode="r",
)

sample_label = np.load(
    all_labels[0],
    mmap_mode="r",
)

assert sample_input.shape == (
    160,
    128,
    128,
    3,
)

assert sample_label.shape == (160,)
assert np.isfinite(sample_input).all()
assert np.isfinite(sample_label).all()

cache_size_bytes = sum(
    path.stat().st_size
    for path in all_inputs + all_labels
)

disk_after = shutil.disk_usage(
    cache_path.parent
)

gc.collect()


# ============================================================
# RESULTS
# ============================================================

print("\n" + "=" * 70)
print("FULL UBFC-rPPG PREPROCESSING RESULT")
print("=" * 70)
print("Recordings processed :", len(ubfc_records))
print("Total input clips    :", len(all_inputs))
print("Total label clips    :", len(all_labels))
print("Training clips       :", split_counts["TRAIN"])
print("Testing clips        :", split_counts["TEST"])
print("Sample input shape   :", sample_input.shape)
print("Sample label shape   :", sample_label.shape)
print("Input dtype          :", sample_input.dtype)
print("Label dtype          :", sample_label.dtype)
print(
    f"Actual cache size    : "
    f"{cache_size_bytes / (1024**3):.2f} GB"
)
print(
    f"Free storage after   : "
    f"{disk_after.free / (1024**3):.2f} GB"
)
print(
    "Train file list      :",
    config.TRAIN.DATA.FILE_LIST_PATH,
)
print(
    "Test file list       :",
    config.TEST.DATA.FILE_LIST_PATH,
)

print("\nComplete UBFC-rPPG preprocessing: PASSED")

=> Merging a config file from /media/data/rPPG/Code/GitHub/Catch_The_Mamba/configs/local/intra/UBFC_RHYTHMMAMBA_LOCAL.yaml
FULL UBFC-rPPG PREPROCESSING
Raw dataset          : /media/data/rPPG/rPPG_Data/Mamba_Hunt/RhythmMamba_DataView/UBFC
Cache directory      : /media/data/rPPG/rPPG_Data/Mamba_Hunt/RhythmMamba_Preprocessed/UBFC/UBFC_SizeW128_SizeH128_ClipLength160_DataTypeStandardized_DataAugNone_LabelTypeStandardized_Crop_faceTrue_Large_boxTrue_Large_size1.5_Dyamic_DetFalse_det_len30_Median_face_boxFalse
Total recordings     : 42
Already completed    : 0
Remaining recordings : 42
Expected clips       : 483
Estimated cache size : 28.30 GB
Free storage before  : 246.93 GB
Parallel processes   : 4
Frame/label mismatch: 0
Preprocessing dataset...


  0%|          | 0/42 [00:00<?, ?it/s]

 40%|████      | 17/42 [01:46<02:05,  5.01s/it]

 43%|████▎     | 18/42 [01:57<02:34,  6.44s/it]

 62%|██████▏   | 26/42 [02:34<01:00,  3.80s/it]

 64%|██████▍   | 27/42 [02:45<01:29,  5.93s/it]

 74%|███████▍  | 31/42 [03:03<00:43,  3.98s/it]

 83%|████████▎ | 35/42 [03:25<00:26,  3.79s/it]

100%|██████████| 42/42 [04:03<00:00,  5.79s/it]



FULL UBFC-rPPG PREPROCESSING RESULT
Recordings processed : 42
Total input clips    : 483
Total label clips    : 483
Training clips       : 342
Testing clips        : 141
Sample input shape   : (160, 128, 128, 3)
Sample label shape   : (160,)
Input dtype          : float64
Label dtype          : float64
Actual cache size    : 28.30 GB
Free storage after   : 218.62 GB
Train file list      : /media/data/rPPG/rPPG_Data/Mamba_Hunt/RhythmMamba_Preprocessed/UBFC/DataFileLists/UBFC_SizeW128_SizeH128_ClipLength160_DataTypeStandardized_DataAugNone_LabelTypeStandardized_Crop_faceTrue_Large_boxTrue_Large_size1.5_Dyamic_DetFalse_det_len30_Median_face_boxFalse_0.0_0.72.csv
Test file list       : /media/data/rPPG/rPPG_Data/Mamba_Hunt/RhythmMamba_Preprocessed/UBFC/DataFileLists/UBFC_SizeW128_SizeH128_ClipLength160_DataTypeStandardized_DataAugNone_LabelTypeStandardized_Crop_faceTrue_Large_boxTrue_Large_size1.5_Dyamic_DetFalse_det_len30_Median_face_boxFalse_0.72_1.0.csv

Complete UBFC-rPPG preprocessin